In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from data_pipeline import cleaning_data

In [3]:
df=pd.read_csv(r"C:\Users\thien\code\AIMY\house-prices-advanced-regression-techniques\train.csv")
target_name='SalePrice'
drop_threshold=0.6

In [4]:
df=df.drop(columns='Id')

In [5]:
X,y=cleaning_data(df,target_name=target_name,drop_threshold=drop_threshold)

In [6]:
X.isna().any(axis=0).sum()

np.int64(0)

In [7]:
X.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'LotShape',
       'LandContour', 'LotConfig', 'Neighborhood', 'Condition1', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinSF2',
       'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir',
       'Electrical', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath',
       'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr',
       'KitchenQual', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageFinish', 'GarageCars', 'GarageQual', 'GarageCond', 'PavedDrive',
       'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
       'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SaleType',
       'SaleC

In [8]:
# ==============================================================================
# QUESTION 1 – House Quality Impact Across Living Area Groups
# ==============================================================================
# Objective : Examine how OverallQual affects SalePrice within different
#             GrLivArea ranges to determine whether quality matters more
#             for small, medium, or large houses.
# ==============================================================================

q1_df = X.copy()
q1_df['logSalePrice'] = y

# chia diện tích thành 4 nhóm
q1_df['AreaGroup'] = pd.qcut(
    q1_df['GrLivArea'],
    q=4,
    labels=['Small', 'Medium', 'Large', 'Very Large']
)

result_q1 = (
    q1_df
    .groupby(['AreaGroup', 'OverallQual'])
    .agg(
        avg_log_price=('logSalePrice', 'mean'),
        house_count=('logSalePrice', 'size')
    )
    .reset_index()
)

result_q1 = result_q1[result_q1['house_count'] >= 5]

print(result_q1.sort_values(
    ['AreaGroup', 'avg_log_price'],
    ascending=[True, False]
).to_string(index=False))

 AreaGroup  OverallQual  avg_log_price  house_count
     Small          7.0      11.851649            7
     Small          6.0      11.761267           81
     Small          5.0      11.712537          189
     Small          4.0      11.479686           70
     Small          3.0      11.188278           12
     Small          2.0      10.814772            5
    Medium          8.0      12.268278           18
    Medium          7.0      12.098385           85
    Medium          6.0      11.927173          117
    Medium          5.0      11.785509          111
    Medium          4.0      11.618849           31
     Large          9.0      12.663086           12
     Large          8.0      12.441802           59
     Large          7.0      12.194537          112
     Large          6.0      12.047520          107
     Large          5.0      11.880091           58
     Large          4.0      11.787654           11
Very Large         10.0      12.932601           17
Very Large  

In [9]:
# ==============================================================================
# QUESTION 2 – Remodeling Premium by House Age
# ==============================================================================
# Objective : Compare average SalePrice between remodeled and non-remodeled
#             homes across different age groups.
# ==============================================================================

q2_df = X.copy()
q2_df['logSalePrice'] = y

q2_df['HouseAge'] = q2_df['YrSold'] - q2_df['YearBuilt']

q2_df['Remodeled'] = (
    q2_df['YearRemodAdd'] > q2_df['YearBuilt']
).astype(int)

q2_df['AgeGroup'] = pd.cut(
    q2_df['HouseAge'],
    bins=[0, 20, 60, 200],
    labels=['<20', '20-60', '60+']
)

result_q2 = (
    q2_df
    .groupby(['AgeGroup', 'Remodeled'])
    .agg(
        avg_log_price=('logSalePrice', 'mean'),
        house_count=('logSalePrice', 'size')
    )
    .reset_index()
)

print(result_q2.to_string(index=False))

AgeGroup  Remodeled  avg_log_price  house_count
     <20          0      12.262688          286
     <20          1      12.384818          214
   20-60          0      11.848184          415
   20-60          1      11.964845          189
     60+          1      11.724084          292


In [10]:
# ==============================================================================
# QUESTION 3 – Neighborhood Ranking by Price and Price Stability
# ==============================================================================
# Objective : Rank neighborhoods using average SalePrice and standard deviation
#             to identify both high-value and stable housing markets.
# ==============================================================================

q3_df = X.copy()
q3_df['logSalePrice'] = y

result_q3 = (
    q3_df
    .groupby('Neighborhood')
    .agg(
        avg_log_price=('logSalePrice', 'mean'),
        std_log_price=('logSalePrice', 'std'),
        house_count=('logSalePrice', 'size')
    )
    .reset_index()
)

result_q3 = result_q3[result_q3['house_count'] >= 10]

result_q3 = result_q3.sort_values(
    'avg_log_price',
    ascending=False
)

print(result_q3.to_string(index=False))

Neighborhood  avg_log_price  std_log_price  house_count
     NoRidge      12.676003       0.289036           41
     NridgHt      12.619415       0.302870           77
     StoneBr      12.585490       0.351960           25
      Timber      12.363460       0.264828           38
     Veenker      12.344180       0.288925           11
     Somerst      12.296500       0.240360           86
     ClearCr      12.239905       0.238445           28
     Crawfor      12.206664       0.324650           51
     Blmngtn      12.169421       0.148151           17
     CollgCr      12.163647       0.254457          150
     Gilbert      12.155809       0.159985           79
      NWAmes      12.130614       0.199408           73
     SawyerW      12.090695       0.312043           59
     Mitchel      11.933954       0.226690           49
       NAmes      11.868052       0.206500          225
       SWISU      11.838442       0.259865           25
      Sawyer      11.811475       0.179552      